In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Outpatient measurement records from hosp.omr for cohort patients.
# Includes weight, height, and eGFR recorded outside of hospital admissions.
# Blood pressure excluded (captured in triage and ed.vitalsign).
# BMI excluded (derived from height and weight).
# Used to supplement ED and inpatient state features with recent baseline measurements.

query_omr = """
SELECT DISTINCT
  o.subject_id,
  o.chartdate,
  o.result_name,
  o.result_value
FROM `physionet-data.mimiciv_3_1_hosp.omr` AS o
WHERE o.subject_id IN (
  SELECT DISTINCT subject_id
  FROM `physionet-data.mimiciv_ed.edstays`
)
AND o.result_name NOT LIKE 'BMI%'
AND o.result_name NOT LIKE 'Blood Pressure%'
AND o.result_name NOT LIKE 'eGFR'
"""

print("Running OMR query...")
df_omr = client.query(query_omr).to_dataframe()
print(f"Shape: {df_omr.shape}")
print(f"Unique subjects: {df_omr['subject_id'].nunique():,}")
print(f"\nResult types:\n{df_omr['result_name'].value_counts()}")
df_omr.head()

In [10]:
indicies = df_omr[df_omr['result_value'] == '.'].index.to_list()

df_omr.drop(index=indicies, inplace=True)

df_omr['result_value'] = df_omr['result_value'].str.strip(">").astype(float)

df_omr['result_name'] = df_omr['result_name'].str.strip(" (Lbs)").str.strip(" (Inches)")

omr_h = df_omr[df_omr['result_name'] == 'Height']
omr_w = df_omr[df_omr['result_name'] == 'Weight']

omr_h.drop_duplicates(inplace=True)
omr_w.drop_duplicates(inplace=True)

In [ ]:
DESCRIPTION = (
    "Height and weight measurement records from hosp.omr for cohort patients — separated to take single measurements of height "
    "as an average of all available height measurements for a patient and retaining all separate weight measurements.  Duplicates were dropped "
    "Used to supplement ED and inpatient state features with recent baseline measurements."
)

ds = Dataset.from_pandas(omr_h, split='height', preserve_index=False)
ds_w = Dataset.from_pandas(omr_w, split='weight', preserve_index=False)

ds.push_to_hub("ADS599-Capstone/raw_data", config_name="height", data_dir="height")
push_dataset_card("ADS599-Capstone/raw_data", config_name="height", split="height", data_dir="height", description=DESCRIPTION)

ds_w.push_to_hub("ADS599-Capstone/raw_data", config_name="weight", data_dir="weight")
push_dataset_card("ADS599-Capstone/raw_data", config_name="weight", split="weight", data_dir="weight", description=DESCRIPTION)

Setting num_proc from 1 back to 1 for the height split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.
